In [1]:
import pandas as pd
import kagglehub
import torch
from transformers import AutoTokenizer, AutoModel

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The cache for model files in Transformers v4.22.0 has been updated. Migrating your old cache. This is a one-time only operation. You can interrupt this and resume the migration later on by calling `transformers.utils.move_cache()`.
0it [00:00, ?it/s]


In [4]:
# read data
df = pd.read_csv("ASAP2_train_sourcetexts.csv")
df.head()

,essay_id,score,full_text,assignment,prompt_name,economically_disadvantaged,student_disability_status,ell_status,race_ethnicity,gender,source_text_1,source_text_2,source_text_3,source_text_4
0,AAAVUP14319000159574,4,The author suggests that studying Venus is wor...,"In ""The Challenge of Exploring Venus,"" the aut...",Exploring Venus,Economically disadvantaged,Identified as having disability,No,Black/African American,F,"The Challenge of Exploring Venus\nVenus, somet...",NaN,NaN,NaN
1,AAAVUP14319000159542,2,NASA is fighting to be alble to to go to Venus...,"In ""The Challenge of Exploring Venus,"" the aut...",Exploring Venus,Not economically disadvantaged,Not identified as having disability,No,Hispanic/Latino,F,"The Challenge of Exploring Venus\nVenus, somet...",NaN,NaN,NaN
2,AAAVUP14319000159461,3,"""The Evening Star"", is one of the brightest po...","In ""The Challenge of Exploring Venus,"" the aut...",Exploring Venus,Economically disadvantaged,Identified as having disability,No,White,M,"The Challenge of Exploring Venus\nVenus, somet...",NaN,NaN,NaN
3,AAAVUP14319000159420,2,The author supports this idea because from rea...,"In ""The Challenge of Exploring Venus,"" the aut...",Exploring Venus,Economically disadvantaged,Not identified as having disability,Yes,Hispanic/Latino,F,"The Challenge of Exploring Venus\nVenus, somet...",NaN,NaN,NaN
4,AAAVUP14319000159419,2,How the author supports this idea is that he s...,"In ""The Challenge of Exploring Venus,"" the aut...",Exploring Venus,Economically disadvantaged,Not identified as having disability,Yes,Hispanic/Latino,M,"The Challenge of Exploring Venus\nVenus, somet...",NaN,NaN,NaN


In [3]:
model_name = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=False)
model = AutoModel.from_pretrained(model_name)
model.eval()  # Set model to inference mode

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/huggingface_hub/file_download.py:943: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


DebertaV2Model(
  (embeddings): DebertaV2Embeddings(
    (word_embeddings): Embedding(128100, 768, padding_idx=0)
    (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
    (dropout): StableDropout()
  )
  (encoder): DebertaV2Encoder(
    (layer): ModuleList(
      (0-11): 12 x DebertaV2Layer(
        (attention): DebertaV2Attention(
          (self): DisentangledSelfAttention(
            (query_proj): Linear(in_features=768, out_features=768, bias=True)
            (key_proj): Linear(in_features=768, out_features=768, bias=True)
            (value_proj): Linear(in_features=768, out_features=768, bias=True)
            (pos_dropout): StableDropout()
            (dropout): StableDropout()
          )
          (output): DebertaV2SelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-07, elementwise_affine=True)
            (dropout): StableDropout()
          )
        )
        (intermedia

In [6]:
def get_embedding(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
        # Mean pooling over token embeddings (ignoring padding)
        attention_mask = inputs["attention_mask"]
        embeddings = outputs.last_hidden_state
        mask_expanded = attention_mask.unsqueeze(-1).expand(embeddings.size()).float()
        summed = torch.sum(embeddings * mask_expanded, dim=1)
        counted = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
        mean_pooled = summed / counted
    return mean_pooled.squeeze().numpy()

In [7]:
df["embedding"] = df["full_text"].apply(get_embedding)
print(df["embedding"].iloc[0].shape)  # Output: (768,)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


(768,)


In [11]:
#df[['embedding']].to_csv("embedding.csv", index=False)

In [20]:
df['embedding'].iloc[0]

array([ 4.76906449e-02,  2.67202482e-02,  2.19494656e-01,  4.51347083e-02,
        7.46750385e-02, -9.00759995e-02,  9.55018774e-02, -6.05992861e-02,
       -3.42350267e-02, -2.04395689e-02,  5.06957620e-02, -2.39427343e-01,
       -7.04513416e-02, -9.67269540e-02,  4.23621535e-01, -3.01077306e-01,
        5.86841665e-02, -9.97908860e-02,  7.84137174e-02,  5.28161786e-02,
       -2.20284134e-01, -2.07829803e-01, -8.61556083e-03,  1.02401204e-01,
       -1.66198477e-01, -5.90712465e-02,  3.29943523e-02,  4.36093062e-02,
       -1.78870603e-01,  1.73247784e-01,  1.92348644e-01,  3.84893343e-02,
       -9.48951989e-02, -9.69724432e-02, -1.09280095e-01,  1.73494667e-02,
       -1.85493112e-01,  1.47956163e-01, -5.44206426e-02,  2.08219662e-01,
       -4.47462127e-02, -1.34106264e-01,  3.23436409e-02, -1.87853798e-01,
        2.94389986e-02,  2.40996651e-05,  2.04909332e-02, -2.14409366e-01,
        1.29802912e-01,  6.05522422e-03,  1.04585752e-01, -8.28055739e-02,
        3.13951135e-01, -